# setup

In [1]:
import os
os.environ["JAVA_HOME"] = "C:\\Program Files\\Eclipse Adoptium\\jdk-17.0.18.8-hotspot"  # adjust version number to match
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .config("spark.driver.memory", "30g") \
    .getOrCreate()

sc = spark.sparkContext
print(sc.version)

4.1.1


In [2]:
import multiprocessing
print(multiprocessing.cpu_count())

8


# 1. read the 2 CSVs using spark

In [3]:
%%time
df_matches = spark.read.options(delimiter=",", header=True, inferSchema=True).csv("battlesStaging_12072020_to_12262020_WL_tagged.csv")

CPU times: total: 15.6 ms
Wall time: 1min 58s


In [4]:
%%time
df_cards = spark.read.options(delimiter=",", header=True, inferSchema=True).csv("CardMasterListSeason18_12082020.csv")

CPU times: total: 0 ns
Wall time: 211 ms


# 2. replace card IDs with card names using spark sql (distributed computing)

In [5]:
%%time
df_matches.createOrReplaceTempView("matches")
df_cards.createOrReplaceTempView("cards")

df_result = spark.sql("""
    SELECT m.*,
        w1.`team.card1.name` as `winner.card1.name`,
        w2.`team.card1.name` as `winner.card2.name`,
        w3.`team.card1.name` as `winner.card3.name`,
        w4.`team.card1.name` as `winner.card4.name`,
        w5.`team.card1.name` as `winner.card5.name`,
        w6.`team.card1.name` as `winner.card6.name`,
        w7.`team.card1.name` as `winner.card7.name`,
        w8.`team.card1.name` as `winner.card8.name`,
        l1.`team.card1.name` as `loser.card1.name`,
        l2.`team.card1.name` as `loser.card2.name`,
        l3.`team.card1.name` as `loser.card3.name`,
        l4.`team.card1.name` as `loser.card4.name`,
        l5.`team.card1.name` as `loser.card5.name`,
        l6.`team.card1.name` as `loser.card6.name`,
        l7.`team.card1.name` as `loser.card7.name`,
        l8.`team.card1.name` as `loser.card8.name`
    FROM matches m
    LEFT JOIN cards w1 ON m.`winner.card1.id` = w1.`team.card1.id`
    LEFT JOIN cards w2 ON m.`winner.card2.id` = w2.`team.card1.id`
    LEFT JOIN cards w3 ON m.`winner.card3.id` = w3.`team.card1.id`
    LEFT JOIN cards w4 ON m.`winner.card4.id` = w4.`team.card1.id`
    LEFT JOIN cards w5 ON m.`winner.card5.id` = w5.`team.card1.id`
    LEFT JOIN cards w6 ON m.`winner.card6.id` = w6.`team.card1.id`
    LEFT JOIN cards w7 ON m.`winner.card7.id` = w7.`team.card1.id`
    LEFT JOIN cards w8 ON m.`winner.card8.id` = w8.`team.card1.id`
    LEFT JOIN cards l1 ON m.`loser.card1.id` = l1.`team.card1.id`
    LEFT JOIN cards l2 ON m.`loser.card2.id` = l2.`team.card1.id`
    LEFT JOIN cards l3 ON m.`loser.card3.id` = l3.`team.card1.id`
    LEFT JOIN cards l4 ON m.`loser.card4.id` = l4.`team.card1.id`
    LEFT JOIN cards l5 ON m.`loser.card5.id` = l5.`team.card1.id`
    LEFT JOIN cards l6 ON m.`loser.card6.id` = l6.`team.card1.id`
    LEFT JOIN cards l7 ON m.`loser.card7.id` = l7.`team.card1.id`
    LEFT JOIN cards l8 ON m.`loser.card8.id` = l8.`team.card1.id`
""")

df_result.cache()
print("Done!", df_result.count(), "rows")
df_result.show(1)

Done! 16795959 rows
+---+-------------------+-----------+-----------+------------------------+----------+-----------------------+-------------------+-------------+-------------------------+------------------------------+---------------+-------------------+----------+----------------------+------------------+------------+------------------------+--------------+------------------+-----------------------------+-------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+---------------+------------------+--------------------+----------------------+------------------+----------------------+------------------+-------------------+-----------------+-----------------+----------------------+---------------------+--------------+-----------------+--------------+-----------------+--------------+--------

# 3. count wins and losses for all cards

In [6]:
%%time
from pyspark.sql.functions import col, lit, sum as spark_sum
from functools import reduce
from pyspark.sql import DataFrame
# Build a union of all winner and loser card columns
#list for dfs
card_dfs = []
#create 16 dataframes one for each column where each instance is a row with 1 win and 0 losses or vice versa
for i in range(1, 9):
    card_dfs.append(
        df_result.select(col(f"`winner.card{i}.name`").alias("card_name"), lit(1).alias("wins"), lit(0).alias("losses"))
    )
    card_dfs.append(
        df_result.select(col(f"`loser.card{i}.name`").alias("card_name"), lit(0).alias("wins"), lit(1).alias("losses"))
    )


#combine all dfs together (stacks all rows)  
all_cards = reduce(DataFrame.union, card_dfs)

# Filter nulls, group and aggregate
df_card_stats = all_cards \
    .filter(col("card_name").isNotNull()) \
    .groupBy("card_name") \
    .agg(
        spark_sum("wins").alias("win_count"),
        spark_sum("losses").alias("loss_count")
    ) \
    .orderBy(col("win_count").desc())
#save
df_card_stats.cache()
df_card_stats.show(10)

+-------------+---------+----------+
|    card_name|win_count|loss_count|
+-------------+---------+----------+
|       Wizard|  5138212|   5141957|
|     Valkyrie|  5039353|   4931289|
|          Zap|  4798476|   4569975|
|Skeleton Army|  4610435|   4568422|
|      The Log|  4458685|   4387700|
|     Fireball|  4156050|   4215116|
|    Hog Rider|  3661370|   3547371|
|  Mega Knight|  3556820|   3536744|
|       Arrows|  3515542|   3476221|
|  Baby Dragon|  3069898|   3026894|
+-------------+---------+----------+
only showing top 10 rows
CPU times: total: 78.1 ms
Wall time: 1min 50s


# 4. add win ratio

In [7]:
%%time
from pyspark.sql.functions import round as spark_round

df_card_stats = df_card_stats.withColumn(
    "win_ratio",
    spark_round(col("win_count") / (col("win_count") + col("loss_count")), 3)
)

df_card_stats.cache()
df_card_stats.show(10)

+-------------+---------+----------+---------+
|    card_name|win_count|loss_count|win_ratio|
+-------------+---------+----------+---------+
|       Wizard|  5138212|   5141957|      0.5|
|     Valkyrie|  5039353|   4931289|    0.505|
|          Zap|  4798476|   4569975|    0.512|
|Skeleton Army|  4610435|   4568422|    0.502|
|      The Log|  4458685|   4387700|    0.504|
|     Fireball|  4156050|   4215116|    0.496|
|    Hog Rider|  3661370|   3547371|    0.508|
|  Mega Knight|  3556820|   3536744|    0.501|
|       Arrows|  3515542|   3476221|    0.503|
|  Baby Dragon|  3069898|   3026894|    0.504|
+-------------+---------+----------+---------+
only showing top 10 rows
CPU times: total: 0 ns
Wall time: 1.15 s


# 5. results

In [8]:
%%time
df_card_stats.orderBy(col("win_ratio").desc()).show(df_card_stats.count(), truncate=False)

+----------------+---------+----------+---------+
|card_name       |win_count|loss_count|win_ratio|
+----------------+---------+----------+---------+
|Mega Minion     |966854   |898319    |0.518    |
|Battle Ram      |509974   |476804    |0.517    |
|Lava Hound      |311339   |292324    |0.516    |
|Night Witch     |938355   |888178    |0.514    |
|Golem           |1160228  |1102175   |0.513    |
|Lightning       |992210   |946387    |0.512    |
|Zap             |4798476  |4569975   |0.512    |
|Rascals         |185359   |177601    |0.511    |
|Bowler          |309926   |296324    |0.511    |
|Flying Machine  |341577   |327391    |0.511    |
|P.E.K.K.A       |1941730  |1861776   |0.511    |
|Goblin Gang     |2318752  |2216543   |0.511    |
|Dark Prince     |1193588  |1148216   |0.51     |
|Mini P.E.K.K.A  |2705738  |2597622   |0.51     |
|Skeleton Barrel |1096119  |1056221   |0.509    |
|Goblin Barrel   |2845695  |2746123   |0.509    |
|Cannon Cart     |266358   |258340    |0.508    |
